In [7]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score, log_loss

file = '/Users/alejandrogomez-paz/Desktop/UFC Project/3. models/logistic_model/features.csv'
df = pd.read_csv(file, index_col=0)
df['date'] = pd.to_datetime(df['date'])

feature_cols = [c for c in df.columns if c.endswith('_diff')]
df = df.dropna(subset=feature_cols)          # drops debut fights (~27%)

cutoff = df['date'].quantile(0.7)
train = df[df['date'] < cutoff]
test  = df[df['date'] >= cutoff]

scaler = StandardScaler().fit(train[feature_cols])   # fit on train only ✓
X_train = scaler.transform(train[feature_cols])
X_test  = scaler.transform(test[feature_cols])
y_train, y_test = train['y'], test['y']

model = LogisticRegression(solver='saga', l1_ratio=0.5, C=1.0, max_iter=100_000)
model.fit(X_train, y_train)

p = model.predict_proba(X_test)[:, 1]
print(f"accuracy: {accuracy_score(y_test, p > 0.5):.3f}")
print(f"AUC:      {roc_auc_score(y_test, p):.3f}")
print(f"log loss: {log_loss(y_test, p):.3f}")

coefs = pd.Series(model.coef_[0], index=feature_cols).sort_values(key=abs, ascending=False)
print(coefs.head(10))

accuracy: 0.614
AUC:      0.657
log loss: 0.654
total_str_attempted_norm_diff   -0.367426
age_diff                        -0.363238
wins_diff                        0.253989
td_attempted_norm_diff           0.202075
losses_diff                     -0.187933
rating_diff                      0.166600
head_attempted_norm_diff         0.149488
head_landed_norm_diff            0.142452
leg_attempted_norm_diff          0.112304
reach_z_diff                     0.109615
dtype: float64
